# Amazon anomaly detection — Kaggle launcher

Attach the **program** source dataset and the Kaggle Amazon-from-space dataset containing `train_v2.csv` and the extracted four-band `train-tif-v2` images. The program source dataset can contain `main.py`, `config.py`, and the package folders directly at its root. Enable a CUDA GPU and see `README.md` for the input requirements and cohort-generation details.

This launcher discovers the source directory under `/kaggle/input`, copies it into writable storage, and runs the program in a fresh Python process. It also supports an attached `project.zip` with either flat or wrapped contents. Dependencies are not installed automatically. The first run creates reproducible development and benchmark manifests in writable storage. The full run performs 1,620 inner fits and 180 outer refits.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import tempfile
import zipfile

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
# Leave None to discover the program dataset, including longer Kaggle mount paths.
# Alternatively, set the directory containing main.py or the path to project.zip.
SOURCE_PROJECT = None
PROJECT = Path('/kaggle/working/comp90051-amazon-anomaly')


def find_projects(directory):
    return sorted({path.parent for path in directory.rglob('main.py')
                   if (path.parent / 'config.py').is_file()
                   and (path.parent / 'training/protocol.py').is_file()
                   and '__MACOSX' not in path.parts})


def choose_project(candidates):
    preferred = [path for path in candidates if path.name == 'program']
    if len(preferred) == 1:
        return preferred[0]
    if len(candidates) == 1:
        return candidates[0]
    raise ValueError(f'Expected one source project; found {candidates}. Set SOURCE_PROJECT explicitly.')


def copy_project(source):
    source = Path(source)
    if not (source / 'main.py').is_file() or not (source / 'training/protocol.py').is_file():
        raise ValueError(f'SOURCE_PROJECT must contain main.py and the package folders: {source}')
    if source.resolve() != PROJECT.resolve():
        shutil.copytree(source, PROJECT, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('__pycache__', '.pytest_cache', '__MACOSX',
                                                     '.DS_Store', '._*', '*.zip'))


if SOURCE_PROJECT is None:
    candidates = find_projects(KAGGLE_INPUT_ROOT)
    if candidates:
        SOURCE_PROJECT = choose_project(candidates)
    else:
        archives = sorted({path for name in ('project.zip', 'comp90051-amazon-anomaly.zip')
                           for path in KAGGLE_INPUT_ROOT.rglob(name)})
        if len(archives) != 1:
            raise ValueError('Attach the program dataset or one project.zip, or set SOURCE_PROJECT explicitly.')
        SOURCE_PROJECT = archives[0]

SOURCE_PROJECT = Path(SOURCE_PROJECT)
if SOURCE_PROJECT.is_file() and SOURCE_PROJECT.suffix.lower() == '.zip':
    with tempfile.TemporaryDirectory(prefix='amazon-source-') as temporary:
        staging = Path(temporary).resolve()
        with zipfile.ZipFile(SOURCE_PROJECT) as archive:
            for member in archive.infolist():
                target = (staging / member.filename).resolve()
                if not target.is_relative_to(staging):
                    raise ValueError('Source ZIP contains a path outside its extraction directory.')
            archive.extractall(staging)
        copy_project(choose_project(find_projects(staging)))
else:
    copy_project(SOURCE_PROJECT)

print(f'Attached source: {SOURCE_PROJECT}')
print(f'Writable program: {PROJECT / "main.py"}')


## Inspect the run configuration

The launcher needs the labelled `train_v2.csv` and extracted four-band TIFFs from the Kaggle dataset. It generates deterministic, disjoint `development_manifest.csv` and `benchmark_manifest.csv` files under `OUTPUT_ROOT / 'cohorts'` on the first run, then reuses them on reruns. Each cohort has the configured R/P/N quotas, and repeated pixel hashes are excluded across cohorts. New cohorts require raw feature extraction; the authenticated cache belongs to a different fixed cohort, so keep `RECOMPUTE_FEATURES = True`.

Dry run prints settings only; the experiment cell performs the data scan and GPU checks. Output is saved under `/kaggle/working`, because attached dataset files are read-only.

Set `MODELS = ['DeepSVDD']` to train only Deep SVDD, or choose `PCA`, `OCSVM`, `LOF`, `MLPAutoencoder`, or `TransformerAE`. Set `MODELS = None` to run all six. A selected model still uses all three feature sets, ten outer folds, three inner folds, and nine image conditions. One model requires 270 inner fits and 30 outer refits.

In [ ]:
MODELS = None  # All six. For one model, use ['DeepSVDD'] (or another model name).
RECOMPUTE_FEATURES = True
MAKE_PLOTS = True
INPUT_ROOT = KAGGLE_INPUT_ROOT  # Includes source, image, and metadata datasets.
OUTPUT_ROOT = PROJECT / 'outputs'
LABELS_PATH = None  # Set an explicit label CSV if multiple copies are attached.

command = [sys.executable, '-u', str(PROJECT / 'main.py'),
           '--input-root', str(INPUT_ROOT), '--output-root', str(OUTPUT_ROOT)]
if MODELS is not None:
    command += ['--models', *MODELS]
if LABELS_PATH is not None:
    command += ['--labels-path', str(LABELS_PATH)]
if RECOMPUTE_FEATURES:
    command += ['--recompute-features']
if not MAKE_PLOTS:
    command += ['--no-plots']
subprocess.run(command + ['--dry-run'], check=True)

## Run the experiment

This cell starts training, streams the program's output, and saves both progress and errors in `OUTPUT_ROOT / 'run.log'`. On failure it includes the final output lines in the displayed exception, so the underlying traceback remains visible.

Rerun with the same source, environment, and output directory to reuse verified completed units after an interruption. The command must include `--models` followed by the selected names when running only a subset; rerun the configuration cell after changing `MODELS`.


In [ ]:
from collections import deque

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
log_path = OUTPUT_ROOT / 'run.log'
last_lines = deque(maxlen=40)
print('Command:', command)
print('Log:', log_path)

with log_path.open('w', encoding='utf-8') as log:
    log.write('Command: ' + json.dumps(command) + '\n')
    log.flush()
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
                last_lines.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise

if returncode:
    raise RuntimeError(f'Program exited with status {returncode}. Full log: {log_path}\n'
                       + ''.join(last_lines))
print('Experiment finished successfully.')


## Inspect and download outputs

Select the most recently finalized evidence archive and display its result tables. Execution completion and the modal-middle numerical requirement are reported separately. See the interpretation section in `README.md` for benchmark limitations.


In [ ]:
import pandas as pd
from IPython.display import FileLink, Image, display

archives = sorted((OUTPUT_ROOT / 'results').glob('*/result_evidence.zip'),
                  key=lambda path: path.stat().st_mtime_ns)
if not archives:
    raise FileNotFoundError('No completed result archive. Inspect the experiment cell above.')
result_directory = archives[-1].parent
completion = json.loads((result_directory / 'completion.json').read_text())
display(pd.Series(completion))
display(pd.read_csv(result_directory / 'summary.csv'))
for figure in sorted((result_directory / 'figures').glob('*.png')):
    display(Image(filename=str(figure)))
display(FileLink(str(result_directory / 'result_evidence.zip')))
print(f'All saved files: {result_directory}')
